# 🏢 CompanyOS — GRPO Training Notebook

**Environment:** CompanyOS on HuggingFace Spaces  
**Method:** GRPO via HuggingFace TRL + Unsloth  


## 0. Detect Runtime (HF Spaces vs Colab)

In [ ]:
import os
import sys

# ── Detect where we're running ─────────────────────────────────────────────
IS_HF_SPACE  = os.path.exists('/home/user')          # HuggingFace Spaces
IS_COLAB     = 'google.colab' in sys.modules         # Google Colab
IS_LOCAL     = not IS_HF_SPACE and not IS_COLAB      # Local machine

print(f'Runtime: {"HF Spaces" if IS_HF_SPACE else "Colab" if IS_COLAB else "Local"}')

# ── Config — edit these ────────────────────────────────────────────────────
ENV_URL       = 'https://satyamshahi-companyos.hf.space'  # your HF Space URL
MODEL_NAME    = 'unsloth/Qwen2.5-1.5B-Instruct'           # small fast model
HF_TOKEN      = os.environ.get('HF_TOKEN', '')            # set in HF Space secrets
WANDB_TOKEN   = os.environ.get('WANDB_API_KEY', '')       # optional

# ── Training hyperparams ───────────────────────────────────────────────────
MAX_EPISODES        = 100
GRPO_UPDATE_EVERY   = 10     # run GRPO update every N episodes
GRPO_BATCH_SIZE     = 4
GRPO_GRAD_ACCUM     = 2
LEARNING_RATE       = 5e-6
MAX_NEW_TOKENS      = 150
LORA_R              = 16

print(f'ENV_URL: {ENV_URL}')
print(f'Model:   {MODEL_NAME}')
print(f'Episodes: {MAX_EPISODES}')

## 1. Install Dependencies

In [ ]:
# ── Install based on runtime ───────────────────────────────────────────────
if IS_HF_SPACE:
    # HF Spaces: use uv for fast installs (pre-installed on most spaces)
    os.system('pip install -q unsloth trl transformers datasets requests matplotlib wandb')
else:
    # Colab / Local
    os.system('pip install -q unsloth trl transformers datasets requests matplotlib wandb')

print('Dependencies installed.')

## 2. Login to HuggingFace (needed for model download + pushing trained model)

In [ ]:
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Logged in to HuggingFace via token.')
else:
    # Interactive login — works on Colab
    print('No HF_TOKEN found. If on HF Spaces, set it in Settings > Secrets.')
    print('If on Colab, run: from huggingface_hub import login; login()')

## 3. Verify Environment is Reachable

In [ ]:
import requests
import json

# ── Health check ───────────────────────────────────────────────────────────
health = requests.get(f'{ENV_URL}/health', timeout=30).json()
print('Health:', health)

# ── Reset test ─────────────────────────────────────────────────────────────
obs = requests.post(f'{ENV_URL}/reset', json={}, timeout=30).json()
print('Task:', obs['observation']['task'])
print('Tools:', list(obs['observation']['tools'].keys()))
print('Progress keys:', list(obs['observation']['progress'].keys()))
print('\n✅ Environment reachable and working!')

## 4. Environment Client

In [ ]:
# ── Simple REST client for the CompanyOS env ──────────────────────────────

def env_reset(task_id=None, seed=None):
    body = {}
    if task_id: body['task_id'] = task_id
    if seed:    body['seed']    = seed
    r = requests.post(f'{ENV_URL}/reset', json=body, timeout=30)
    return r.json()['observation']

def env_step(app, method, params=None):
    body = {'app': app, 'method': method, 'params': params or {}}
    r = requests.post(f'{ENV_URL}/step', json=body, timeout=30)
    d = r.json()
    return d['observation'], d['reward'], d['done'], d['info']

print('Environment client ready.')

## 5. Random Baseline — BEFORE Training

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

RANDOM_ACTIONS = [
    ('ticketdesk',    'list_tickets',    {}),
    ('ticketdesk',    'search_tickets',  {'query': 'vendor'}),
    ('ticketdesk',    'get_ticket',      {'ticket_id': 'T-001'}),
    ('ticketdesk',    'update_ticket',   {'ticket_id': 'T-001', 'field': 'priority',  'value': 'high'}),
    ('ticketdesk',    'update_ticket',   {'ticket_id': 'T-001', 'field': 'verified',  'value': True}),
    ('ticketdesk',    'update_ticket',   {'ticket_id': 'T-003', 'field': 'status',    'value': 'open'}),
    ('datahub',       'list_metrics',    {}),
    ('datahub',       'query_metric',    {'metric_name': 'vendor_compliance_score'}),
    ('datahub',       'refresh_data',    {'metric_name': 'vendor_compliance_score'}),
    ('datahub',       'get_approver',    {'role': 'CFO'}),
    ('datahub',       'get_approver',    {'role': 'CFO_DELEGATE'}),
    ('approvalflow',  'list_approval_types', {}),
    ('approvalflow',  'submit_approval', {'approval_type': 'vendor_onboarding', 'approver': 'david.kim',
                                          'data': {'vendor_name': 'ACME', 'compliance_score': 85}}),
    ('approvalflow',  'check_status',    {'approval_id': 'APR-001'}),
    # Bad actions — random agent doesn't know better
    ('ticketdesk',    'update_ticket',   {'ticket_id': 'T-003', 'field': 'status', 'value': 'closed'}),
    ('approvalflow',  'submit_approval', {'approval_type': 'vendor_onboarding', 'approver': 'sarah.chen',
                                          'data': {'vendor_name': 'ACME', 'compliance_score': 85}}),
]

BASELINE_EPISODES = 100
baseline_rewards   = []
baseline_successes = []

print(f'Running random baseline for {BASELINE_EPISODES} episodes...')
for ep in range(BASELINE_EPISODES):
    obs      = env_reset()
    done     = False
    ep_reward = 0.0
    while not done:
        app, method, params = random.choice(RANDOM_ACTIONS)
        _, reward, done, info = env_step(app, method, params)
        ep_reward += reward
    baseline_rewards.append(ep_reward)
    baseline_successes.append(info.get('success', False))
    if (ep + 1) % 20 == 0:
        mean_r = sum(baseline_rewards[-20:]) / 20
        sr     = sum(baseline_successes[-20:]) / 20
        print(f'  Ep {ep+1:>4} | avg_reward={mean_r:+.2f} | success_rate={sr:.0%}')

print(f'\n── Baseline Results ─────────────────────')
print(f'  Mean reward:  {sum(baseline_rewards)/len(baseline_rewards):+.2f}')
print(f'  Success rate: {sum(baseline_successes)/len(baseline_successes):.1%}')
print(f'  Best episode: {max(baseline_rewards):+.2f}')

## 6. Load Model with Unsloth

In [ ]:
import os
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'

import torch
from unsloth import FastLanguageModel

print(f'Loading {MODEL_NAME}...')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow!)"}')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit   = True,   # saves VRAM — works on T4 and A10G
    token          = HF_TOKEN if HF_TOKEN else None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    target_modules = ['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_alpha     = LORA_R,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',  # saves memory
)

print(f'Model loaded. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 7. Agent: Observation → Action via LLM

In [ ]:
"""
- Suppresses max_new_tokens/max_length warning
- Reduces max_new_tokens (JSON actions are short, 150 is overkill)
- Uses greedy decoding by default (faster, do_sample only when exploring)
"""

import re

SYSTEM_PROMPT = """You are an enterprise workflow agent navigating CompanyOS.
Complete the given task by calling tools across three apps:

ticketdesk:   list_tickets | search_tickets(query) | get_ticket(ticket_id) | update_ticket(ticket_id, field, value)
datahub:      list_metrics | query_metric(metric_name) | refresh_data(metric_name) | get_approver(role)
approvalflow: list_approval_types | submit_approval(approval_type, approver, data) | check_status(approval_id) | escalate(approval_id, reason)

Key rules:
- Always refresh stale data before using it
- Check get_approver() before submitting approvals — the CFO may be OOO
- Verify a ticket before linking an approval
- Poll check_status() until the approval resolves

Respond ONLY with a single JSON object. No explanation. No markdown. Example:
{"app": "ticketdesk", "method": "get_ticket", "params": {"ticket_id": "T-001"}}"""

def obs_to_prompt(obs):
    return (
        f"Task: {obs['task']}\n"
        f"Step: {obs['step']}/{obs['max_steps']} (remaining: {obs['steps_remaining']})\n"
        f"Progress: {json.dumps(obs['progress'])}\n"
        f"Last result: {json.dumps(obs.get('last_result', {}))}\n"
        f"\nWhat is your next action?"
    )

def parse_action(text):
    """Parse JSON action from model output. Falls back to safe default."""
    try:
        match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
        if match:
            action = json.loads(match.group())
            if action.get('app') and action.get('method'):
                return action
    except Exception:
        pass
    return {'app': 'ticketdesk', 'method': 'list_tickets', 'params': {}}

# ── Suppress the max_length warning once ───────────────────────────────────
model.generation_config.max_length = None

def agent_act(obs, temperature=0.7):
    """Sample an action from the model given the current observation."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': obs_to_prompt(obs)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            inputs,
            max_new_tokens  = 80,          # JSON actions are ~40-60 tokens
            temperature     = temperature if temperature > 0 else None,
            do_sample       = temperature > 0,
            top_p           = 0.9 if temperature > 0 else None,
            pad_token_id    = tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    return parse_action(text), text

print('Agent ready.')


## 8. Supervised Fine-tuning

In [ ]:
"""
Runs a SCRIPTED EXPERT that actually solves each task, then SFTs
the model on those perfect trajectories.

This teaches the model:
  1. The correct JSON action format
  2. How to read the observation and pick task-appropriate actions
  3. The correct sequence of steps to complete each task type
"""

import random
import copy

# ── Scripted expert: solves any CompanyOS task using the task description ───

# Maps task keywords → (ticket_id, metric_name, approval_type, approver_role)
TASK_PROFILES = {
    'vendor onboarding': {
        'ticket_id': 'T-001',
        'metric_name': 'vendor_compliance_score',
        'approval_type': 'vendor_onboarding',
        'approver_role': 'CFO_DELEGATE',
        'approval_data': {'vendor_name': 'ACME', 'compliance_score': 85},
        'needs_refresh': True,
        'needs_priority': True,
    },
    'expense report': {
        'ticket_id': 'T-002',
        'metric_name': 'expense_report_447_amount',
        'approval_type': 'expense_report',
        'approver_role': 'CFO_DELEGATE',
        'approval_data': {'amount': 4350.00, 'employee_id': 'john.doe', 'report_id': '447'},
        'needs_refresh': False,
        'needs_priority': False,
    },
    'bug': {
        'ticket_id': 'T-003',
        'metric_name': 'auth_service_error_rate',
        'approval_type': 'bug_escalation',
        'approver_role': 'CTO',
        'approval_data': {'ticket_id': 'T-003', 'error_rate': 18.5},
        'needs_refresh': False,
        'needs_priority': False,
        'needs_unblock': True,
        'needs_assignee': True,
    },
    'license': {
        'ticket_id': 'T-004',
        'metric_name': 'software_license_expiry_days',
        'approval_type': 'license_renewal',
        'approver_role': 'PROCUREMENT',
        'approval_data': {'license_name': 'Q3 Software License', 'expiry_days': 12},
        'needs_refresh': False,
        'needs_priority': False,
    },
    'handbook': {
        'ticket_id': 'T-005',
        'metric_name': 'employee_count_hr_dept',
        'approval_type': 'handbook_update',
        'approver_role': 'HR_LEAD',
        'approval_data': {'section': '4', 'change_summary': 'Policy update for section 4'},
        'needs_refresh': True,
        'needs_priority': True,
    },
}

# Approver names by role
APPROVER_NAMES = {
    'CFO_DELEGATE': 'david.kim',
    'CTO': 'mike.ross',
    'HR_LEAD': 'priya.nair',
    'PROCUREMENT': 'tom.baker',
}

def identify_task(task_description):
    """Match the task description to a profile."""
    desc_lower = task_description.lower()
    for keyword, profile in TASK_PROFILES.items():
        if keyword in desc_lower:
            return profile
    return TASK_PROFILES['vendor onboarding']  # fallback

def expert_policy(obs):
    """Given an observation, return the optimal next action."""
    task_desc = obs['task']
    progress  = obs['progress']
    last      = obs.get('last_result', {})
    profile   = identify_task(task_desc)

    tid  = profile['ticket_id']
    met  = profile['metric_name']
    atyp = profile['approval_type']
    role = profile['approver_role']
    data = profile['approval_data']

    # Step 1: Verify ticket (if not done)
    if 'ticket_verified' in progress and not progress['ticket_verified']:
        return {'app': 'ticketdesk', 'method': 'update_ticket',
                'params': {'ticket_id': tid, 'field': 'verified', 'value': True}}

    # Step 1b: Set priority (if needed)
    if 'ticket_priority_set' in progress and not progress['ticket_priority_set']:
        return {'app': 'ticketdesk', 'method': 'update_ticket',
                'params': {'ticket_id': tid, 'field': 'priority', 'value': 'high'}}

    # Step 1c: Unblock ticket (bug task)
    if 'ticket_unblocked' in progress and not progress['ticket_unblocked']:
        return {'app': 'ticketdesk', 'method': 'update_ticket',
                'params': {'ticket_id': tid, 'field': 'status', 'value': 'open'}}

    # Step 1d: Set assignee (bug task)
    if 'ticket_assignee_set' in progress and not progress['ticket_assignee_set']:
        return {'app': 'ticketdesk', 'method': 'update_ticket',
                'params': {'ticket_id': tid, 'field': 'assignee', 'value': 'engineering-team'}}

    # Step 2: Refresh stale data (if needed for this task)
    if 'metric_refreshed' in progress and not progress['metric_refreshed']:
        return {'app': 'datahub', 'method': 'refresh_data',
                'params': {'metric_name': met}}

    # Step 3: Query metric
    if 'metric_queried' in progress and not progress['metric_queried']:
        return {'app': 'datahub', 'method': 'query_metric',
                'params': {'metric_name': met}}

    # Step 4: Get approver
    if 'approval_submitted' in progress and not progress['approval_submitted']:
        # First check if we know the approver
        if isinstance(last, dict) and last.get('approver'):
            # We have the approver, submit
            approver = APPROVER_NAMES.get(role, 'david.kim')
            return {'app': 'approvalflow', 'method': 'submit_approval',
                    'params': {'approval_type': atyp, 'approver': approver, 'data': data}}
        else:
            # Look up approver first
            return {'app': 'datahub', 'method': 'get_approver',
                    'params': {'role': role}}

    # Step 5: Poll approval status
    if 'approval_approved' in progress and not progress['approval_approved']:
        # Find the approval ID from last result
        if isinstance(last, dict) and last.get('approval_id'):
            return {'app': 'approvalflow', 'method': 'check_status',
                    'params': {'approval_id': last['approval_id']}}
        # If we don't have ID, list approvals
        return {'app': 'approvalflow', 'method': 'list_approvals', 'params': {}}

    # Fallback: list tickets (shouldn't reach here)
    return {'app': 'ticketdesk', 'method': 'list_tickets', 'params': {}}


# ── Run expert episodes and collect trajectories ───────────────────────────
EXPERT_EPISODES = 30   # 5 tasks × 6 runs each
expert_data = []       # (prompt, completion, episode_reward)

print(f'Running {EXPERT_EPISODES} expert episodes...')

expert_rewards = []
expert_successes = 0

for ep in range(EXPERT_EPISODES):
    obs  = env_reset()
    done = False
    ep_pairs  = []
    ep_reward = 0.0

    while not done:
        prompt = obs_to_prompt(obs)
        action = expert_policy(obs)
        action_json = json.dumps(action)

        obs, reward, done, info = env_step(
            action['app'], action['method'], action.get('params', {})
        )
        ep_pairs.append((prompt, action_json))
        ep_reward += reward

    expert_rewards.append(ep_reward)
    if info.get('success'):
        expert_successes += 1
        # Store ALL steps from successful episodes
        for p, c in ep_pairs:
            expert_data.append((p, c, ep_reward))
    elif ep_reward > 0:
        # Also keep high-reward partial completions
        for p, c in ep_pairs:
            expert_data.append((p, c, ep_reward))

mean_r = sum(expert_rewards) / len(expert_rewards)
print(f'Expert: {expert_successes}/{EXPERT_EPISODES} success | mean_reward={mean_r:+.2f}')
print(f'Collected {len(expert_data)} SFT examples from successful/good episodes')

if len(expert_data) < 20:
    print('⚠️  Few successful episodes. Adding all episodes for format learning...')
    for ep_idx in range(EXPERT_EPISODES):
        obs = env_reset()
        done = False
        while not done:
            prompt = obs_to_prompt(obs)
            action = expert_policy(obs)
            action_json = json.dumps(action)
            obs, reward, done, info = env_step(
                action['app'], action['method'], action.get('params', {})
            )
            expert_data.append((prompt, action_json, 0.0))
    print(f'Now have {len(expert_data)} SFT examples')

# ── SFT on expert trajectories ────────────────────────────────────────────
model.train()
sft_optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
)

SFT_EPOCHS = 10

for epoch in range(SFT_EPOCHS):
    total_loss = 0.0
    indices = list(range(len(expert_data)))
    random.shuffle(indices)

    for step_i, idx in enumerate(indices):
        prompt_text, completion_text, _ = expert_data[idx]

        # Prompt-only tokens (for masking)
        prompt_msgs = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt_text},
        ]
        prompt_ids = tokenizer.apply_chat_template(
            prompt_msgs, tokenize=True,
            add_generation_prompt=True,
            return_tensors='pt',
        ).to(model.device)

        # Full sequence
        full_msgs = [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': prompt_text},
            {'role': 'assistant', 'content': completion_text},
        ]
        full_ids = tokenizer.apply_chat_template(
            full_msgs, tokenize=True, return_tensors='pt',
        ).to(model.device)

        # Mask prompt tokens
        labels = full_ids.clone()
        labels[0, :prompt_ids.shape[-1]] = -100

        outputs = model(full_ids, labels=labels)
        loss = outputs.loss / 4
        loss.backward()

        if (step_i + 1) % 4 == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            sft_optimizer.step()
            sft_optimizer.zero_grad()

        total_loss += loss.item() * 4

    # Flush
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    sft_optimizer.step()
    sft_optimizer.zero_grad()

    avg_loss = total_loss / len(expert_data)
    print(f'  SFT Epoch {epoch+1}/{SFT_EPOCHS} | loss={avg_loss:.4f}')

model.eval()

# ── Quick test ─────────────────────────────────────────────────────────────
print('\nTest on each task type:')
for task_id in ['task_vendor_onboarding', 'task_expense_report', 'task_bug_escalation',
                'task_license_renewal', 'task_handbook_update']:
    test_obs = env_reset(task_id=task_id)
    test_action, test_raw = agent_act(test_obs, temperature=0.1)
    print(f'  {task_id}: {json.dumps(test_action)}')

print('\n✅ Expert SFT warmup complete!')


## 9. GRPO Training Loop

In [ ]:
"""
- Log-prob on completion tokens only (prompt masked with -100)
- Episode returns for advantages
- Epsilon-greedy exploration (mix random valid actions to keep getting positive signal)
- Debug logging of sample actions
"""

import os
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'

import warnings
warnings.filterwarnings('ignore')

import time

# ── Training state ─────────────────────────────────────────────────────────
train_rewards    = []
train_successes  = []
train_steps      = []
grpo_losses      = []
shortcut_counts  = []

# Buffer: list of (prompt, completion, episode_return) tuples
trajectory_buffer = []

# ── Exploration schedule ───────────────────────────────────────────────────
EPSILON_START = 0.3     # 30% random at start (model still learning format)
EPSILON_END   = 0.05    # 5% random at end
def get_epsilon(ep):
    return EPSILON_START - (EPSILON_START - EPSILON_END) * min(ep / MAX_EPISODES, 1.0)

# ── Optimizer (created once, reused across all updates) ────────────────────
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
)

print(f'Starting REINFORCE training for {MAX_EPISODES} episodes...')
print(f'Policy update every {GRPO_UPDATE_EVERY} episodes')
print(f'Epsilon: {EPSILON_START} → {EPSILON_END}')
print('─' * 60)

start_time = time.time()

for episode in range(MAX_EPISODES):
    model.eval()

    obs           = env_reset()
    done          = False
    ep_reward     = 0.0
    ep_shortcuts  = 0
    ep_prompts, ep_completions, ep_rewards = [], [], []
    epsilon = get_epsilon(episode)

    while not done:
        prompt = obs_to_prompt(obs)

        # ── Epsilon-greedy: sometimes use random valid actions ──────────
        if random.random() < epsilon:
            app, method, params = random.choice(RANDOM_ACTIONS)
            action = {'app': app, 'method': method, 'params': params}
            completion_text = json.dumps(action)
        else:
            action, completion_text = agent_act(obs)
            completion_text = json.dumps(action)  # ensure valid JSON

        obs, reward, done, info = env_step(
            action.get('app',    'ticketdesk'),
            action.get('method', 'list_tickets'),
            action.get('params', {})
        )

        ep_prompts.append(prompt)
        ep_completions.append(completion_text)
        ep_rewards.append(reward)
        ep_reward   += reward
        ep_shortcuts = info.get('shortcut_attempts', 0)

    train_rewards.append(ep_reward)
    train_successes.append(info.get('success', False))
    train_steps.append(info.get('step', 0))
    shortcut_counts.append(ep_shortcuts)

    # Store each step with TOTAL episode return
    for p, c in zip(ep_prompts, ep_completions):
        trajectory_buffer.append((p, c, ep_reward))

    # ── REINFORCE policy update every N episodes ───────────────────────────
    if (episode + 1) % GRPO_UPDATE_EVERY == 0 and len(trajectory_buffer) >= GRPO_BATCH_SIZE:
        model.train()

        batch_size = min(200, len(trajectory_buffer))
        batch = trajectory_buffer[-batch_size:]

        # Normalize episode returns to get advantages
        returns = torch.tensor([t[2] for t in batch], dtype=torch.float32)
        advantages = (returns - returns.mean()) / (returns.std() + 1e-8)

        optimizer.zero_grad()
        total_loss = 0.0
        n_updates  = 0

        for i in range(0, batch_size, GRPO_BATCH_SIZE):
            mb_end  = min(i + GRPO_BATCH_SIZE, batch_size)
            mb_loss = torch.tensor(0.0, device=model.device, requires_grad=True)

            for j in range(i, mb_end):
                prompt_text, completion_text, _ = batch[j]

                # ── Prompt-only tokens (for masking) ───────────────
                prompt_msgs = [
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': prompt_text},
                ]
                prompt_ids = tokenizer.apply_chat_template(
                    prompt_msgs, tokenize=True,
                    add_generation_prompt=True,
                    return_tensors='pt'
                ).to(model.device)
                prompt_len = prompt_ids.shape[-1]

                # ── Full sequence (prompt + completion) ────────────
                full_msgs = [
                    {'role': 'system',    'content': SYSTEM_PROMPT},
                    {'role': 'user',      'content': prompt_text},
                    {'role': 'assistant', 'content': completion_text},
                ]
                full_ids = tokenizer.apply_chat_template(
                    full_msgs, tokenize=True, return_tensors='pt'
                ).to(model.device)

                # ── Mask: -100 for prompt, real ids for completion ──
                labels = full_ids.clone()
                labels[0, :prompt_len] = -100

                outputs  = model(full_ids, labels=labels)
                log_prob = -outputs.loss

                # Policy gradient: loss = −advantage × log π(a|s)
                mb_loss = mb_loss + (-advantages[j].to(model.device) * log_prob)

            mb_loss = mb_loss / (mb_end - i)
            mb_loss.backward()

            n_updates  += 1
            total_loss += mb_loss.item()

            if n_updates % GRPO_GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()

        # Flush remaining gradients
        if n_updates % GRPO_GRAD_ACCUM != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        grpo_losses.append(total_loss / max(n_updates, 1))

        # Clear trajectory buffer
        trajectory_buffer.clear()

        model.eval()

        mean_r  = sum(train_rewards[-GRPO_UPDATE_EVERY:])   / GRPO_UPDATE_EVERY
        sr      = sum(train_successes[-GRPO_UPDATE_EVERY:]) / GRPO_UPDATE_EVERY
        mean_sc = sum(shortcut_counts[-GRPO_UPDATE_EVERY:]) / GRPO_UPDATE_EVERY
        elapsed = (time.time() - start_time) / 60

        print(
            f'Ep {episode+1:>4} | '
            f'avg_reward={mean_r:+6.2f} | '
            f'success={sr:.0%} | '
            f'shortcuts={mean_sc:.1f} | '
            f'loss={grpo_losses[-1]:.4f} | '
            f'eps={epsilon:.2f} | '
            f'{elapsed:.1f}min'
        )

        # Debug: show sample action from last episode
        if ep_completions:
            print(f'    sample: {ep_completions[0]}')

        if episode > 50 and mean_sc > 5:
            print(f'  ⚠️  high shortcut rate ({mean_sc:.1f})')

print('\nTraining complete!')


## 10. Learning Curves 📈

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def smooth(data, window=10):
    if len(data) < window:
        return data
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig = plt.figure(figsize=(20, 10))
fig.suptitle('CompanyOS — Training Results', fontsize=15, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.35)  # 2x4 not 2x3

WINDOW = 10
ep_range_smooth = range(WINDOW - 1, len(train_rewards))

# ── Plot 1: Reward curve — baseline vs trained (spans all 4 cols) ───────────
ax1 = fig.add_subplot(gs[0, :])
baseline_mean = sum(baseline_rewards) / len(baseline_rewards)
baseline_std  = np.std(baseline_rewards)
ax1.axhspan(baseline_mean - baseline_std, baseline_mean + baseline_std,
            alpha=0.15, color='tomato', label='Random baseline range')
ax1.axhline(baseline_mean, color='tomato', linestyle='--', linewidth=1.5,
            label=f'Random baseline mean ({baseline_mean:.1f})')
ax1.plot(train_rewards, alpha=0.2, color='steelblue')
ax1.plot(list(ep_range_smooth), smooth(train_rewards, WINDOW),
         color='steelblue', linewidth=2.5, label=f'Trained agent (smoothed w={WINDOW})')
ax1.set_title('Episode Reward: Random Baseline vs Trained Agent', fontsize=12)
ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Reward')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# ── Plot 2: Success rate ─────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
success_vals = [1.0 if s else 0.0 for s in train_successes]
baseline_sr  = sum(1 for s in baseline_successes if s) / len(baseline_successes)
ax2.axhline(baseline_sr, color='tomato', linestyle='--', linewidth=1.5,
            label=f'Baseline SR ({baseline_sr:.1%})')
if len(success_vals) >= WINDOW:
    ax2.plot(list(ep_range_smooth), smooth(success_vals, WINDOW),
             color='seagreen', linewidth=2)
ax2.set_title('Task Success Rate')
ax2.set_xlabel('Episode')
ax2.set_ylabel('Success Rate')
ax2.set_ylim(-0.05, 1.05)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# ── Plot 3: Steps per episode ────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(train_steps, alpha=0.2, color='mediumpurple')
if len(train_steps) >= WINDOW:
    ax3.plot(list(ep_range_smooth), smooth(train_steps, WINDOW),
             color='mediumpurple', linewidth=2)
ax3.set_title('Steps per Episode\n(lower = more efficient)')
ax3.set_xlabel('Episode')
ax3.set_ylabel('Steps')
ax3.grid(True, alpha=0.3)

# ── Plot 4: GRPO loss ────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
if grpo_losses:
    ax4.plot(grpo_losses, color='darkorange', linewidth=2, marker='o', markersize=4)
    ax4.set_title('GRPO Training Loss')
    ax4.set_xlabel(f'Update (every {GRPO_UPDATE_EVERY} episodes)')
    ax4.set_ylabel('Loss')
    ax4.grid(True, alpha=0.3)
else:
    ax4.text(0.5, 0.5, 'No GRPO updates yet', ha='center', va='center',
             transform=ax4.transAxes)
    ax4.set_title('GRPO Training Loss')

# ── Plot 5: Shortcut attempts ────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 3])
if shortcut_counts:
    ax5.plot(shortcut_counts, alpha=0.3, color='crimson')
    if len(shortcut_counts) >= WINDOW:
        sc_range = range(WINDOW - 1, len(shortcut_counts))
        ax5.plot(list(sc_range), smooth(shortcut_counts, WINDOW),
                 color='crimson', linewidth=2)
    ax5.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax5.set_title('Shortcut Attempts per Episode\n(should trend → 0)')
    ax5.set_xlabel('Episode')
    ax5.set_ylabel('Attempts')
    ax5.grid(True, alpha=0.3)
else:
    ax5.text(0.5, 0.5, 'No shortcuts tracked yet', ha='center', va='center',
             transform=ax5.transAxes)
    ax5.set_title('Shortcut Attempts')

plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → learning_curves.png')

## 11. Before vs After Summary Table

In [ ]:
# ── Summary stats ──────────────────────────────────────────────────────────
last_n = min(50, len(train_rewards))
trained_mean = sum(train_rewards[-last_n:]) / last_n
trained_sr   = sum(train_successes[-last_n:]) / last_n

print('=' * 50)
print('       BEFORE vs AFTER TRAINING')
print('=' * 50)
print(f'  {"Metric":<25} {"Random":>8}  {"Trained":>8}')
print('-' * 50)
print(f'  {"Mean reward":<25} {baseline_mean:>+8.2f}  {trained_mean:>+8.2f}')
print(f'  {"Success rate":<25} {baseline_sr:>8.1%}  {trained_sr:>8.1%}')
print(f'  {"Improvement (reward)":<25} {"":>8}  {trained_mean - baseline_mean:>+8.2f}')
print(f'  {"Improvement (SR)":<25} {"":>8}  {trained_sr - baseline_sr:>+8.1%}')
print('=' * 50)

## 12. Save & Push Trained Model to HF Hub

In [ ]:
# ── Save locally ───────────────────────────────────────────────────────────
model.save_pretrained('companyos-agent')
tokenizer.save_pretrained('companyos-agent')
print('Model saved locally to ./companyos-agent')

# ── Push to HF Hub (needs HF_TOKEN with write access) ─────────────────────
HF_USERNAME = 'satyamshahi'   # ← your HF username

if HF_TOKEN:
    model.push_to_hub(f'{HF_USERNAME}/companyos-agent', token=HF_TOKEN)
    tokenizer.push_to_hub(f'{HF_USERNAME}/companyos-agent', token=HF_TOKEN)
    print(f'Pushed to https://huggingface.co/{HF_USERNAME}/companyos-agent')
else:
    print('Set HF_TOKEN to push to hub.')

## 12. Quick Eval — Watch Trained Agent Play an Episode

In [ ]:
# ── Watch one episode with the trained agent ───────────────────────────────
print('=== TRAINED AGENT EPISODE ===')
obs   = env_reset(task_id='task_vendor_onboarding')
done  = False
total = 0.0

print(f'Task: {obs["task"]}\n')

while not done:
    action, raw = agent_act(obs, temperature=0.3)  # lower temp for eval
    obs, reward, done, info = env_step(
        action.get('app', ''),
        action.get('method', ''),
        action.get('params', {})
    )
    total += reward
    status = '✅' if reward > 0 else '❌'
    print(f'  Step {info["step"]:>2} | {action.get("app","")}.{action.get("method","")} → r={reward:+.2f} {status}')
    print(f'          Progress: {info["progress"]}')

outcome = '🎉 SUCCESS' if info.get('success') else '⏱️ TIMEOUT'
print(f'\n{outcome} | Total reward: {total:.2f}')